In [2]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from boxmot import BotSort
from pathlib import Path
import warnings

In [4]:
MODEL_PATH = './bestmodel.keras'
VIDEO_PATH = "video.mp4"
IMG_SIZE = (256, 256)

In [5]:
model = tf.keras.models.load_model(MODEL_PATH,compile=False)

I0000 00:00:1758289753.528692     323 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5580 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
def preprocess_frame_for_deeplab(frame, img_size):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_tensor = tf.convert_to_tensor(img_rgb, dtype=tf.float32)
    img_resized = tf.image.resize(img_tensor, img_size)
    img_preprocessed = preprocess_input(img_resized)
    img_for_prediction = tf.expand_dims(img_preprocessed, axis=0)
    return img_for_prediction

In [7]:
def process_video(video_path, segmentation_model, botsort_tracker,
                  show_id=True,
                  save_output=True,
                  tracked_video_name="tracked_lanes_output.mp4",
                  mask_video_name="segmentation_mask_output.mp4",
                  display_windows=True,
                  show_boxes=True):

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    out_tracked = None
    out_mask = None
    if save_output:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_tracked = cv2.VideoWriter(tracked_video_name, fourcc, fps, (frame_width, frame_height))
        out_mask = cv2.VideoWriter(mask_video_name, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        input_tensor = preprocess_frame_for_deeplab(frame, IMG_SIZE)
        prediction_logits = segmentation_model.predict(input_tensor, verbose=0)
        output_mask_tensor = tf.argmax(prediction_logits, axis=-1)
        output_mask = tf.squeeze(output_mask_tensor, axis=0).numpy().astype(np.uint8)
        output_mask_resized = cv2.resize(output_mask, (frame_width, frame_height), interpolation=cv2.INTER_NEAREST)
        
        contours, _ = cv2.findContours(output_mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        detections_for_botsort = []
        for cnt in contours:
            if cv2.contourArea(cnt) > 600:
                x, y, w_box, h_box = cv2.boundingRect(cnt)
                detections_for_botsort.append([x, y, x + w_box, y + h_box, 0.95, 0])

        detections = np.array(detections_for_botsort) if len(detections_for_botsort) > 0 else np.empty((0, 6))
        tracks = botsort_tracker.update(detections, frame)

        color_mask = np.zeros_like(frame)
        color_mask[output_mask_resized == 1] = [0, 0, 255]
        mask_display_frame = cv2.addWeighted(frame, 0.7, color_mask, 0.3, 0)
        final_frame = mask_display_frame.copy()

        if show_boxes:
            for track in tracks:
                if not isinstance(track, np.ndarray):
                    continue
                
                x1, y1, x2, y2, track_id, _, _, _ = track
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                track_id = int(track_id)

                cv2.rectangle(final_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                if show_id:
                    cv2.putText(final_frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        if save_output:
            out_tracked.write(final_frame)
            out_mask.write(mask_display_frame)

        if display_windows:
            cv2.imshow("Lane Tracking and Segmentation", final_frame)
            if cv2.waitKey(1) == ord('q'):
                break

    cap.release()
    if save_output:
        out_tracked.release()
        out_mask.release()
    if display_windows:
        cv2.destroyAllWindows()

In [8]:
reid_weights_path = Path('osnet_x1_0_msmt17.pt')

tracker = BotSort(
    reid_weights=reid_weights_path,
    device=0, 
    half=False,
    with_reid=True
)

2025-09-19 17:53:32.662 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.12 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
2025-09-19 17:53:32.664 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at osnet_x1_0_msmt17.pt; skipping download.
2025-09-19 17:53:32.930 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from osnet_x1_0_msmt17.pt


In [13]:
process_video(
    video_path=VIDEO_PATH,
    segmentation_model=model,
    botsort_tracker=tracker,
    show_id=True,
    save_output=True,
    tracked_video_name="final_tracked_lanes_botsort.mp4", 
    mask_video_name="final_segmentation_mask_botsort.mp4", 
    display_windows=False,
    show_boxes=True
)